# Phase 1 Group C: LLM SFT 实验

> 基于 Qwen3-0.6B/1.5B 大语言模型，使用 LoRA 进行参数高效微调
>
> **任务**: DimASR (给定文本 + Aspect，预测 VA 分数)
> **数据集**: zho_restaurant (繁体中文餐厅评论)

# 一、环境配置

In [ ]:
# 安装 LLM 相关依赖
!pip install peft transformers bitsandbytes accelerate -q

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1.2 路径配置

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/kaggle/working")
DATA_ROOT = Path("/kaggle/input/datasets/kintsugi0v0/dimabsa")

TRAIN_FILE = DATA_ROOT / "zho_restaurant_train_alltasks.jsonl"
DEV_FILE = DATA_ROOT / "zho_restaurant_dev_task1.jsonl"
TEST_FILE = DATA_ROOT / "zho_restaurant_test_task1.jsonl"

print(f"Data path: {DATA_ROOT}")
print(f"Train: {TRAIN_FILE.exists()}, Dev: {DEV_FILE.exists()}, Test: {TEST_FILE.exists()}")

# 二、数据加载与预处理

In [ ]:
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

def explode_quadruplet(data):
    exploded = []
    for item in data:
        for q in item.get('Quadruplet', []):
            try:
                v, a = map(float, q['VA'].split('#'))
                exploded.append({
                    'Text': item['Text'],
                    'Aspect': q.get('Aspect', ''),
                    'Valence': v,
                    'Arousal': a
                })
            except:
                continue
    return exploded

def explode_aspect_va(data):
    exploded = []
    for item in data:
        for av in item.get('Aspect_VA', []):
            try:
                v, a = map(float, av['VA'].split('#'))
                exploded.append({
                    'Text': item['Text'],
                    'Aspect': av.get('Aspect', ''),
                    'Valence': v,
                    'Arousal': a
                })
            except:
                continue
    return exploded

print("工具函数定义完成")

# 三、超参数配置

In [ ]:
# 超参数配置
CONFIG = {
    'seed': 42,
    'max_length': 256,
    'batch_size': 2,
    'gradient_accumulation_steps': 8,
    'learning_rate': 1e-4,
    'epochs': 3,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'dropout': 0.1,
}

# Group C 实验配置
MODEL_CONFIGS = {
    'Exp-C1': {'model': 'Qwen/Qwen3-0.6B', 'type': 'sft_lora'},
    'Exp-C2': {'model': 'Qwen/Qwen3-1.7B', 'type': 'sft_lora'},
    'Exp-C3': {'model': 'Qwen/Qwen3-0.6B', 'type': 'regression_lora'},
    'Exp-C4': {'model': 'Qwen/Qwen3-1.7B', 'type': 'regression_lora'},
}

print("配置完成!")
print(f"可用实验: {list(MODEL_CONFIGS.keys())}")

# 四、数据格式转换

In [ ]:
def convert_to_sft_format(data, tokenizer, max_length=256):
    """
    将数据转换为 SFT 格式
    """
    sft_data = []
    
    for item in data:
        instruction = f"给定文本「{item['Text']}」和方面词「{item['Aspect']}」，预测VA分数。"
        response = f"Valence={item['Valence']:.2f}, Arousal={item['Arousal']:.2f}"
        
        messages = [
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": response}
        ]
        
        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        sft_data.append({
            'input': input_text,
            'output': response,
            'text': item['Text'],
            'aspect': item['Aspect'],
            'valence': item['Valence'],
            'arousal': item['Arousal']
        })
    
    return sft_data

def convert_to_regression_format(data):
    """
    转换为回归格式
    """
    converted = []
    
    for item in data:
        prompt = f"文本: {item['Text']}\n方面词: {item['Aspect']}\n预测VA分数:"
        converted.append({
            'prompt': prompt,
            'text': item['Text'],
            'aspect': item['Aspect'],
            'valence': item['Valence'],
            'arousal': item['Arousal']
        })
    
    return converted

# 五、数据集定义

In [ ]:
class DimASRSFTDataset(Dataset):
    """SFT 数据集"""
    
    def __init__(self, data, tokenizer, max_length=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        encoding = self.tokenizer(
            item['input'],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': encoding['input_ids'].squeeze(0).clone()
        }


class DimASRRegressionDataset(Dataset):
    """回归数据集"""
    
    def __init__(self, data, tokenizer, max_length=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        encoding = self.tokenizer(
            item['prompt'],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor([item['valence'], item['arousal']], dtype=torch.float)
        }

print("数据集类定义完成")

# 六、模型定义

In [ ]:
def create_model(model_type, model_name, dropout=0.1):
    """创建 LLM 模型"""
    
    # 量化配置
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16  # 使用 bfloat16
    )
    
    # LoRA 配置
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=dropout,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        bias='none',
        task_type=TaskType.CAUSAL_LM
    )
    
    # 加载 tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    if model_type == 'sft_lora':
        # SFT + LoRA
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.bfloat16
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()
        
        return model, tokenizer
    
    elif model_type == 'regression_lora':
        # 回归 + LoRA
        base_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.bfloat16
        )
        model = get_peft_model(base_model, lora_config)
        
        # 冻结非 LoRA 参数
        for name, param in model.named_parameters():
            if 'lora' not in name.lower():
                param.requires_grad = False
        
        # 添加回归头 - 使用模型的 dtype
        hidden_size = model.config.hidden_size
        model.regressor = nn.Linear(hidden_size, 2)
        # 确保 dtype 与模型一致
        model.regressor = model.regressor.to(dtype=torch.bfloat16)
        model.regressor = model.regressor.to(device=next(model.parameters()).device)
        
        return model, tokenizer
    
    else:
        raise ValueError(f"Unknown model type: {model_type}")

# 七、训练与评估函数

In [ ]:
def train_epoch_sft(model, dataloader, optimizer, device, gradient_accumulation_steps=4):
    """SFT 训练 - 用于 sft_lora 模式"""
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    for step, batch in enumerate(tqdm(dataloader, desc="训练")):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss / gradient_accumulation_steps
        loss.backward()
        
        if (step + 1) % gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * gradient_accumulation_steps
    
    return total_loss / len(dataloader)


def train_epoch_regression(model, dataloader, optimizer, device, criterion):
    """回归训练 - 用于 regression_lora 模式"""
    model.train()
    total_loss = 0
    
    for batch in tqdm(dataloader, desc="训练"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        # 获取 hidden states
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden = outputs.hidden_states[-1][:, -1, :]
        
        # 转换 dtype 以匹配模型
        hidden = hidden.to(model.dtype)
        
        # 回归预测
        va_pred = model.regressor(hidden)
        va_pred = torch.clamp(va_pred, min=1.0, max=9.0)
        
        # 关键：labels 转换为 model dtype，避免 backward 时 dtype 不一致
        labels = labels.to(model.dtype)
        
        loss = criterion(va_pred, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate_model(model, dataloader, device, criterion, model_type='regression_lora'):
    """评估模型 - 支持 sft_lora 和 regression_lora"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch in tqdm(dataloader, desc="评估"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        if model_type == 'regression_lora':
            # 回归模式：直接使用 hidden state 和 regressor
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
            hidden = outputs.hidden_states[-1][:, -1, :]
            hidden = hidden.to(model.dtype)
            va_pred = model.regressor(hidden)
        else:
            # SFT 模式：使用 hidden state + 临时 regressor
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
            hidden = outputs.hidden_states[-1][:, -1, :]
            hidden = hidden.to(model.dtype)
            # 创建一个临时 regressor（复用 model.regressor 如果存在）
            if hasattr(model, 'regressor'):
                va_pred = model.regressor(hidden)
            else:
                # 如果没有 regressor，使用均值作为预测（fallback）
                va_pred = torch.full_like(labels, 5.5)
        
        va_pred = torch.clamp(va_pred, min=1.0, max=9.0)
        
        # labels 转换为 model dtype
        labels = labels.to(model.dtype)
        
        loss = criterion(va_pred, labels)
        
        total_loss += loss.item()
        # 转回 float32 再转 numpy（numpy 不支持 bfloat16）
        all_preds.append(va_pred.float().cpu())
        all_labels.append(labels.float().cpu())
    
    preds = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    rmse = np.sqrt(np.mean((preds - labels) ** 2))
    
    return total_loss / len(dataloader), rmse, preds, labels

print("训练和评估函数定义完成")

# 八、主训练函数

In [ ]:
def main(exp_name, model_name, model_type):
    """运行单个实验"""
    set_seed(CONFIG['seed'])
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"{'='*50}")
    print(f"实验: {exp_name}")
    print(f"模型: {model_name} ({model_type})")
    print(f"设备: {device}")
    print(f"{'='*50}")
    
    # 加载数据
    print("\n[1/5] 加载数据...")
    train_data = load_jsonl(DATA_ROOT / "zho_restaurant_train_alltasks.jsonl")
    dev_data = load_jsonl(DATA_ROOT / "zho_restaurant_dev_task1.jsonl")
    
    train_exploded = explode_quadruplet(train_data)
    dev_exploded = explode_aspect_va(dev_data)
    print(f"训练集: {len(train_exploded)} 样本")
    print(f"验证集: {len(dev_exploded)} 样本")
    
    # 创建模型
    print("\n[2/5] 创建模型...")
    model, tokenizer = create_model(model_type, model_name, CONFIG.get('dropout', 0.1))
    model.to(device)
    
    # 计算参数量
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"可训练参数量: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")
    print(f"总参数量: {total_params:,}")
    
    # 准备数据
    print("\n[3/5] 准备数据...")
    if model_type == 'sft_lora':
        train_formatted = convert_to_sft_format(train_exploded, tokenizer, CONFIG['max_length'])
        dev_formatted = convert_to_sft_format(dev_exploded, tokenizer, CONFIG['max_length'])
        train_dataset = DimASRSFTDataset(train_formatted, tokenizer, CONFIG['max_length'])
        dev_dataset = DimASRSFTDataset(dev_formatted, tokenizer, CONFIG['max_length'])
    else:
        train_formatted = convert_to_regression_format(train_exploded)
        dev_formatted = convert_to_regression_format(dev_exploded)
        train_dataset = DimASRRegressionDataset(train_formatted, tokenizer, CONFIG['max_length'])
        dev_dataset = DimASRRegressionDataset(dev_formatted, tokenizer, CONFIG['max_length'])
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
    dev_loader = DataLoader(dev_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
    
    # 优化器
    print("\n[4/5] 配置优化器...")
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG['learning_rate'],
        weight_decay=CONFIG['weight_decay']
    )
    criterion = nn.MSELoss()
    
    # 训练
    print("\n[5/5] 开始训练...")
    best_rmse = float('inf')
    best_model_state = None
    
    for epoch in range(CONFIG['epochs']):
        # 根据模型类型选择训练函数
        if model_type == 'regression_lora':
            train_loss = train_epoch_regression(model, train_loader, optimizer, device, criterion)
        else:
            train_loss = train_epoch_sft(
                model, train_loader, optimizer, device,
                CONFIG['gradient_accumulation_steps']
            )
        
        # 评估
        dev_loss, dev_rmse, preds, labels = evaluate_model(
            model, dev_loader, device, criterion, model_type
        )
        
        print(f"Epoch {epoch+1}/{CONFIG['epochs']}: "
              f"训练损失={train_loss:.4f} | 验证RMSE={dev_rmse:.4f}")
        
        if epoch == 0:
            print(f"  [调试] 预测值范围: [{preds.min():.2f}, {preds.max():.2f}]")
            print(f"  [调试] 真实值范围: [{labels.min():.2f}, {labels.max():.2f}]")
        
        if dev_rmse < best_rmse:
            best_rmse = dev_rmse
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"  ✓ 最佳模型 (RMSE: {best_rmse:.4f})")
    
    # 保存
    torch.save(best_model_state, PROJECT_ROOT / f"{exp_name}_best.pt")
    
    results = {
        'exp_name': exp_name,
        'model_name': model_name,
        'model_type': model_type,
        'best_rmse': float(best_rmse),
        'trainable_params': int(trainable_params),
        'total_params': int(total_params)
    }
    with open(PROJECT_ROOT / f"{exp_name}_results.json", 'w') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    
    print(f"\n{'='*50}")
    print(f"完成! 最佳 RMSE: {best_rmse:.4f}")
    print(f"{'='*50}")
    
    return best_rmse

# 九、运行实验

In [ ]:
# 运行单个实验
EXP_NAME = 'Exp-C1' 

config = MODEL_CONFIGS[EXP_NAME]
main(EXP_NAME, config['model'], config['type'])

# 十、批量运行所有实验

In [ ]:
for exp_name, config in MODEL_CONFIGS.items():
    print(f"\n{'#'*60}")
    print(f"运行 {exp_name}")
    print(f"{'#'*60}")
    main(exp_name, config['model'], config['type'])

# 十一、结果汇总

In [ ]:
# 结果汇总
results = []
for exp in ['Exp-C1', 'Exp-C2', 'Exp-C3', 'Exp-C4']:
    with open(PROJECT_ROOT / f"{exp}_results.json") as f:
        r = json.load(f)
        results.append({
            '实验': exp,
            '方法': r['model_type'].upper(),
            'RMSE': round(r['best_rmse'], 4),
            '参数量': f"{r['total_params']/1e9:.1f}B",
            '可训练参数': f"{r['trainable_params']/r['total_params']*100:.2f}%"
        })

df = pd.DataFrame(results)
print("=" * 70)
print("Phase 1 Group C (LLM SFT) 实验结果汇总")
print("=" * 70)
print(df.to_string(index=False))

df.to_csv(PROJECT_ROOT / "phase1_groupC_results.csv", index=False, encoding='utf-8-sig')

# 十二、可视化

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ACL_COLORS = {
    'blue': '#3175B0',
    'orange': '#E69C2E',
    'red': '#CC3333',
    'green': '#4C8C2B',
}

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 14})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 图1: RMSE 对比
x = np.arange(len(df))
bars = ax1.bar(df['实验'], df['RMSE'], color=ACL_COLORS['blue'], alpha=0.8)
for bar, rmse in zip(bars, df['RMSE']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{rmse:.3f}', ha='center', va='bottom', fontsize=10)

ax1.set_ylabel('RMSE')
ax1.set_title('Phase 1 Group C: LLM SFT Results')
ax1.set_ylim(0, 1.0)
ax1.tick_params(axis='x', rotation=15)

# 图2: 模型规模对比
ax2.bar(df['实验'], [float(p.replace('B','')) for p in df['参数量']], 
        color=ACL_COLORS['orange'], alpha=0.8)
ax2.set_ylabel('Parameters (B)')
ax2.set_title('Model Size')
ax2.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "phase1_groupC_results.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"图片已保存至: {PROJECT_ROOT / 'phase1_groupC_results.png'}")